# De Bruijn graphs

You will be implementing one of the primary assembly algorithms from short-read data that is used today. We will implement a simple form of the algorithm where we _assume perfect sequencing_. That is, everything is sequenced exactly once and there are no errors or variants in the sequencing. 

A graph is composed of **nodes** and **edges** and we will need to develop a data strcture to track edges between nodes in our graph. We have provided the basic class structure as well as descriptions of functions to `add_edge` and `remove_edge` from the graph. You will need to implement these functions in order to then build the de Bruijn graph. 

In our implementation below, we use a `defaultdict` data structure to hold a list of all edges in the graph where all "right" nodes connected to a "left" node are stored in a list for that node.

```
build_debruijn_graph:
define substring length k and input string
For each k-length substring of input:
  split k mer into left and right k-1 mer
  add k-1 mers as nodes with a directed edge from left k-1 mer to right k-1 mer
```

---
## Eulerian walk

To continue our implementation from last class, we will use our De Bruijn graph to output a valid sequence from the assembly. This is implemented as a recursive algorithm by considering all valid edges. You will notice that as you change $k$, we are able to better recapitulate our sequence depending on how repetitive it is. In a more complex implementation of a Eulerian walk there are heuristics and defined rules for determining the validity of traversing a specific edge in the graph to result in a full graph-traversal. One of these methods is to traverse the graph in a depth first manner to avoid sectioning off any part of the graph in the traversal. In our implementation we will ignore these for simplicity.

```
eulerian_walk:
Beginning at first_node as node

For node:
    follow a random valid edge from node
    remove edge
    recurse
```


In [104]:
def read_fastq(filename):
    """Read sequences from a FASTQ file.

    Args:
        filename (str): Path to FASTQ file.

    Returns:
        list: List of DNA sequence strings (quality scores ignored).

    Example:
        >>> reads = read_fastq("test_reads.fastq")
        >>> len(reads)
        10
    """
    sequences = []
    with open(filename, 'r') as f:
        line_count = 0
        for num, line in enumerate(f):
            line_count += 1
            if line_count % 4 == 2:  # Sequence line in FASTQ format
                sequences.append(line.strip())

            if num == (100000 * 4):
                break
    return sequences

In [89]:
"""De Bruijn Graph Genome Assembly Module.

This module provides classes and functions for constructing De Bruijn graphs
from sequencing reads and performing genome assembly using Eulerian path
traversal.
"""

from collections import defaultdict
import random
import copy


class DeBruijnGraph:
    """Main class for De Bruijn graphs and genome assembly.

    This class builds De Bruijn graphs from sequencing reads and performs
    genome assembly by finding Eulerian paths through connected components
    of the graph.

    Attributes:
        graph (defaultdict): Adjacency list representation of graph edges.
            Keys are (k-1)-mers, values are lists of adjacent (k-1)-mers.
        k (int): The k-mer size used for graph construction.

    Example:
        >>> reads = ["ATGGCGTACG", "GCGTACGTTA", "ACGTTACCAT"]
        >>> dbg = DeBruijnGraph(reads, k=6)
        >>> contigs = dbg.assemble_contigs(seed=42)
        >>> len(contigs) > 0
        True
    """

    def __init__(self, reads, k):
        """Initialize De Bruijn graph from sequencing reads.

        Args:
            reads (list): List of DNA sequence strings.
            k (int): K-mer size for graph construction.

        Example:
            >>> reads = ["ATGGCG", "GCGTGC", "TGCAAC"]
            >>> dbg = DeBruijnGraph(reads, k=4)
            >>> len(dbg.graph) > 0
            True
        """
        self.graph = defaultdict(list)
        self.incounts = defaultdict(int)
        self.k = k
        self.build_graph_from_reads(reads, k)
        self.start_nodes = []
        self.build_starts()

    def build_starts(self):
        '''Build start site list'''
        for node in self.graph.keys():
            if (self.incounts[node] == 0) or (len(self.graph[node]) - self.incounts[node] == 1):
                self.start_nodes.append(node)
        #print("We found the following start nodes:")
        #pprint(self.start_nodes)
        

    def add_edge(self, left, right):
        """Add a directed edge to the graph.

        Args:
            left (str): Source (k-1)-mer node.
            right (str): Destination (k-1)-mer node.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.add_edge("ATG", "TGG")
            >>> "TGG" in dbg.graph["ATG"]
            True
        """
        if not right in self.graph[left]:
            self.graph[left].append(right)
            self.incounts[right] += 1


    def remove_edge(self, left, right):
        """Remove a directed edge from the graph.

        Args:
            left (str): Source (k-1)-mer node.
            right (str): Destination (k-1)-mer node.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.add_edge("ATG", "TGG")
            >>> dbg.remove_edge("ATG", "TGG")
            >>> len(dbg.graph["ATG"])
            0
        """
        if right in self.graph[left]:
            self.graph[left].remove(right)

    def build_graph_from_reads(self, reads, k):
        """Build De Bruijn graph from multiple sequencing reads.

        Extracts all k-mers from all reads and adds edges between
        consecutive (k-1)-mers within each k-mer.

        Args:
            reads (list): List of DNA sequence strings.
            k (int): K-mer length for graph construction.

        Example:
            >>> reads = ["ATGGC", "TGGCA"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.build_graph_from_reads(reads, 4)
            >>> "ATG" in dbg.graph
            True
        """
        # iterate1 evevry read
        for num, read in enumerate(reads):
            if num % 1000 == 0:
                print(f"\rread {num}", end = '', flush=True)
            # how much k-mers can we cut out of this read?
            num_kmers = len(read) - k + 1
            
            # build a graph for this read by cutting out all k-mers and connecting their (k-1)-mers
            for i in range(num_kmers):
                # cut out a k-mer
                kmer = read[i : i + k]
                
                # get the left and right (k-1)-mers from this k-mer
                left_k_minus_1 = kmer[:-1]  
                right_k_minus_1 = kmer[1:]  
                
                # add an edge from the left (k-1)-mer to the right (k-1)-mer in the graph
                self.add_edge(left_k_minus_1, right_k_minus_1)

    def eulerian_walk(self, node, graph, seed=None):
        """Perform recursive Eulerian walk on a graph component.

        This is a recursive function that follows all edges from a node
        to traverse the graph, building a path in reverse order.

        Args:
            node (str): Current node to traverse from.
            graph (defaultdict): Graph or subgraph to traverse.
            seed (int, optional): Seed for random edge selection.

        Returns:
            list: List of (k-1)-mers traversed (in reverse order).

        Example:
            >>> reads = ["ATGGCG"]
            >>> dbg = DeBruijnGraph(reads, k=4)
            >>> graph_copy = defaultdict(list, dbg.graph)
            >>> tour = dbg.eulerian_walk("ATG", graph_copy, seed=42)
            >>> len(tour) > 0
            True
        """
        # set random seed for reproducibility if provided
        if seed is not None:
            random.seed(seed)
            
        tour = [] 
        
        # while there are still edges to follow from this node
        debug1 = graph[node]
        debug2 = node
        while len(graph[node]) > 0:
            # follow a random valid edge from node
            next_node = random.choice(graph[node])
            
            # remove edge
            graph[node].remove(next_node)
            
            # recurse
            # this will follow all edges from next_node until it hits a dead end, returning the path taken
            sub_tour = self.eulerian_walk(next_node, graph, seed=None)
            
            # add the path taken from next_node to the tour
            tour.extend(sub_tour)
            
        # add the current node to the tour after all edges have been followed
        tour.append(node)
        
        return tour

    def assemble_contigs(self, seed=None):
        """Assemble all contigs from the De Bruijn graph.

        Finds all connected components and generates an Eulerian path
        for each component, producing multiple assembled contigs.

        Args:
            seed (int, optional): Random seed for reproducible assembly.

        Returns:
            list: List of assembled contig sequences (DNA strings).

        Example:
            >>> reads = ["ATGGCGTACG", "GCGTACGTTA", "ACGTTACCAT"]
            >>> dbg = DeBruijnGraph(reads, k=6)
            >>> contigs = dbg.assemble_contigs(seed=42)
            >>> all(isinstance(c, str) for c in contigs)
            True
        """

        # build a list to hold all the contigs we will assemble
        contigs = []
        
        # because we will be modifying the graph by removing edges as we walk through it,
        # We build a working copy of the graph so that we don't mess up the original graph structure for other tours or future operations,

        working_graph = defaultdict(list)
        for node, edges in self.graph.items():
            working_graph[node] = edges.copy()
        
        #working_graph = copy.deepcopy(self.graph)
        
        count = 1
        total = len(self.start_nodes)
        # find connected components and generate a tour for each one until all edges have been used up
        for start_node in self.start_nodes:
            # iterate through all nodes in the working graph to find a node that still has edges to follow
            #for node in list(working_graph.keys()):
            if len(working_graph[start_node]) > 0:
                if (count % 5000) == 0:
                    print(f"\rWorking on start node {count} out of {total}", end = '', flush=True)
                count += 1
                """if len(working_graph[node]) > 0:
                    start_node = node 
                    break # as soon as we find a node with edges, we can break out of the loop and start a tour from that node"""
            
            # end condition: 
            # if we went through all nodes and didn't find any with edges, 
            # start_node will keep None。
            # it means we've used up all edges in the graph and we're done assembling contigs
            # if start_node is None:
            #     break # break while loop
                
            # main function:eulerian walk and tour to sequence for this component
            # use the start_node we found to start an Eulerian walk on the working graph, which will give us a tour of (k-1)-mers in reverse order
                tour = self.eulerian_walk(start_node, working_graph, seed)
            
            # reverse the tour to get the correct order of nodes and convert it into a DNA sequence using the tour_to_sequence function
                sequence = self.tour_to_sequence(tour)
            
            # if we got a non-empty sequence from this tour, it means we successfully assembled a contig, so we add it to our list of contigs
                if sequence:
                    contigs.append(sequence)
                
        # after we've found tours for all connected components and assembled all contigs, we return the list of contig sequences
        return contigs
        

    def tour_to_sequence(self, tour):
        """Convert a tour of (k-1)-mers into a DNA sequence.

        Args:
            tour (list): List of (k-1)-mer strings in order.

        Returns:
            str: Assembled DNA sequence.

        Example:
            >>> dbg = DeBruijnGraph([], k=4)
            >>> tour = ['ATG', 'TGG', 'GGC', 'GCG']
            >>> dbg.tour_to_sequence(tour)
            'ATGGCG'
        """

        #in case we didn't find any tour, we should return an empty string instead of IndexError
        if not tour:
            return ""

            
        # reverse the tour to get the correct order of nodes
        forward_tour = tour[::-1]        

            
        # find the first node in the tour, which will be the starting point of our sequence
        #sequence = forward_tour[0]
        

        # from second node onwards, we will stitch together the sequence by adding the last character of each node
        #for node in forward_tour[1:]:
            
          
            # since each node is a (k-1)-mer, the last character of the node is the next character in the sequence that we need to add
            #sequence += node[-1]

        sequence = forward_tour[0] + ''.join(node[-1] for node in forward_tour[1:])
                    
        # after stitching together all nodes in the tour, we will have the full assembled sequence for this contig
        return sequence

    def get_assembly_stats(self, contigs):
        """Calculate assembly statistics for assembled contigs.

        Args:
            contigs (list): List of contig sequences.

        Returns:
            dict: Dictionary containing assembly statistics:
                - num_contigs: Total number of contigs
                - total_length: Total assembled sequence length
                - longest_contig: Length of longest contig
                - shortest_contig: Length of shortest contig
                - mean_length: Mean contig length
                - n50: N50 statistic

        Example:
            >>> contigs = ["ATGGCG", "TTTAAA", "CCCCCCCCCC"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> stats = dbg.get_assembly_stats(contigs)
            >>> stats['num_contigs']
            3
        """
        # if we didn't assemble any contigs, we should return 0 for all stats to avoid errors and make it clear that the assembly was empty
        if not contigs:
            return {
                'num_contigs': 0, 'total_length': 0,
                'longest_contig': 0, 'shortest_contig': 0,
                'mean_length': 0, 'n50': 0
            }

        # calculate basic stats like number of contigs, total length, longest and shortest contig, mean length
        lengths = [len(c) for c in contigs]
        
        num_contigs = len(lengths)
        total_length = sum(lengths)
        longest_contig = max(lengths)
        shortest_contig = min(lengths)
        mean_length = total_length / num_contigs
        
        # caculate N50
        # order the contig lengths from longest to shortest
        lengths.sort(reverse=True)
        
        n50 = 0
        running_sum = 0
        half_total = total_length / 2.0
        
        # iterate through the ordered lengths and keep a running sum until we reach at least half of the total length   
        for length in lengths:
            running_sum += length
            # when accumulated length reaches at least half of the total, the current contig length is the N50
            if running_sum >= half_total:
                n50 = length
                break

        # return all the result as dict
        return {
            'num_contigs': num_contigs,
            'total_length': total_length,
            'longest_contig': longest_contig,
            'shortest_contig': shortest_contig,
            'mean_length': mean_length,
            'n50': n50
        }

    def write_fasta(self, contigs, filename):
        """Write assembled contigs to a FASTA file.

        Args:
            contigs (list): List of contig sequences.
            filename (str): Output FASTA filename.

        Example:
            >>> contigs = ["ATGGCG", "TTTAAA"]
            >>> dbg = DeBruijnGraph([], k=4)
            >>> dbg.write_fasta(contigs, "output.fasta")
        """
        # write mode
        with open(filename, 'w') as f:
            # iterate through all contigs and write them to the file in FASTA format
            for i, contig in enumerate(contigs):
                # wirte name : >contig_1, >contig_2 ...
                f.write(f">contig_{i+1}\n")
                # write sequence
                f.write(f"{contig}\n")


---
# Toy Example

In [87]:
print("="*60)
print("EXAMPLE 1: Assembling 9 overlapping reads into one contig")
print("="*60)
from pprint import pprint
toy_reads_1 = [
    "XTGGCGTACG",
    "ATGGCGTACG",  # Read 1
    "GGCGTACGTT",  # Read 2: overlaps with Read 1
    "CGTACGTTAC",  # Read 3: overlaps with Read 2
    "TACGTTACCA",  # Read 4: overlaps with Read 3
    "CGTTACCATG",  # Read 5: overlaps with Read 4
    "TTACCATGGG",  # Read 6: overlaps with Read 5
    "ACCATGGGCC",  # Read 7: overlaps with Read 6
    "CATGGGCCTA",  # Read 8: overlaps with Read 7
    "TGGGCCTAAA",   # Read 9: overlaps with Read 8
    "TTACCPTGGG",  # Read 6: overlaps with Read 5
    "ACCPTGGGCC",  # Read 7: overlaps with Read 6
    "CPTGGGCCTA",  # Read 8: overlaps with Read 7    
    "TGGGCCTAAA",   # Read 9: overlaps with Read 8
    "ACCPTGFGCC",  # Read 7: overlaps with Read 6
    "CPTGFGCCTA",  # Read 8: overlaps with Read 7    
    "TGFGCCTAAA",   # Read 9: overlaps with Read 8

]

print(f"\nInput: {len(toy_reads_1)} reads")
print("First read:  ", toy_reads_1[0])
print("Last read:   ", toy_reads_1[-1])
print(f"\nBuilding De Bruijn graph with k=9...")
deb = DeBruijnGraph(toy_reads_1, 9)
pprint(deb.graph)
pprint(deb.incounts)
print(deb.assemble_contigs())

EXAMPLE 1: Assembling 9 overlapping reads into one contig

Input: 17 reads
First read:   XTGGCGTACG
Last read:    TGFGCCTAAA

Building De Bruijn graph with k=9...
read 0defaultdict(<class 'list'>,
            {'ACCATGGG': ['CCATGGGC'],
             'ACCPTGFG': ['CCPTGFGC'],
             'ACCPTGGG': ['CCPTGGGC'],
             'ACGTTACC': ['CGTTACCA'],
             'ATGGCGTA': ['TGGCGTAC'],
             'ATGGGCCT': ['TGGGCCTA'],
             'CATGGGCC': ['ATGGGCCT'],
             'CCATGGGC': ['CATGGGCC'],
             'CCPTGFGC': ['CPTGFGCC'],
             'CCPTGGGC': ['CPTGGGCC'],
             'CGTACGTT': ['GTACGTTA'],
             'CGTTACCA': ['GTTACCAT'],
             'CPTGFGCC': ['PTGFGCCT'],
             'CPTGGGCC': ['PTGGGCCT'],
             'GCGTACGT': ['CGTACGTT'],
             'GFGCCTAA': ['FGCCTAAA'],
             'GGCGTACG': ['GCGTACGT'],
             'GGGCCTAA': ['GGCCTAAA'],
             'GTACGTTA': ['TACGTTAC'],
             'GTTACCAT': ['TTACCATG'],
             'PTGFGCCT'

---
# Real data
This section utilizes a FASTQ file with 10 million "perfect" 150 bp reads simulated
from the mouse genome (`GRCm39`) and tasks your program to ingest, assemble, and traverse
the genome with statistical output report.

In [105]:
"""Driver program for mouse genome assembly using De Bruijn graphs.

This script demonstrates how to use the completed DeBruijnGraph class to
assemble 10 million perfect 150bp single-end reads from the mouse genome.

Usage:
    Run all cells in order in a Jupyter notebook.

Expected input:
    - File: mouse_SE_150bp.fq
    - Format: FASTQ
    - Reads: 10 million perfect 150bp single-end reads
    - Source: Simulated from mouse genome using wgsim

Output:
    - mouse_assembly.fasta: Assembled contigs
    - mouse_assembly_stats.txt: Detailed statistics
"""

import time
from datetime import datetime


def write_statistics_file(
    stats_file,
    input_file,
    num_reads,
    avg_read_length,
    k_mer_size,
    random_seed,
    num_nodes,
    num_edges,
    stats,
    contig_lengths,
    timing,
    coverage_estimate,
    assembly_fraction
):
    """Write comprehensive assembly statistics to text file.
    
    Args:
        stats_file (str): Output filename.
        input_file (str): Input FASTQ filename.
        num_reads (int): Number of reads processed.
        avg_read_length (float): Average read length.
        k_mer_size (int): K-mer size used.
        random_seed (int): Random seed used.
        num_nodes (int): Number of graph nodes.
        num_edges (int): Number of graph edges.
        stats (dict): Assembly statistics.
        contig_lengths (list): Sorted list of contig lengths.
        timing (dict): Timing information.
        coverage_estimate (float): Estimated sequencing coverage.
        assembly_fraction (float): Assembly size as % of genome.
    """
    with open(stats_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("MOUSE GENOME ASSEMBLY STATISTICS\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Input File: {input_file}\n")
        f.write(f"K-mer Size: {k_mer_size}\n")
        f.write(f"Random Seed: {random_seed}\n\n")
        
        f.write("-"*80 + "\n")
        f.write("INPUT DATA\n")
        f.write("-"*80 + "\n")
        f.write(f"Number of reads:         {num_reads:,}\n")
        f.write(f"Average read length:     {avg_read_length:.1f} bp\n")
        f.write(f"Total sequencing data:   {num_reads * avg_read_length:,.0f} bp\n")
        f.write(f"Estimated coverage:      {coverage_estimate:.1f}x\n")
        f.write(f"Read time:               {timing['read_time']:.2f} seconds\n\n")
        
        f.write("-"*80 + "\n")
        f.write("DE BRUIJN GRAPH CONSTRUCTION\n")
        f.write("-"*80 + "\n")
        f.write(f"Graph nodes:             {num_nodes:,}\n")
        f.write(f"Graph edges:             {num_edges:,}\n")
        f.write(f"Average out-degree:      {num_edges/num_nodes:.2f}\n")
        f.write(f"Construction time:       {timing['graph_time']:.2f} seconds\n\n")
        
        f.write("-"*80 + "\n")
        f.write("ASSEMBLY RESULTS\n")
        f.write("-"*80 + "\n")
        f.write(f"Number of contigs:       {stats['num_contigs']:,}\n")
        f.write(f"Total assembly length:   {stats['total_length']:,} bp\n")
        f.write(f"Assembly vs. genome:     {assembly_fraction:.2f}%\n")
        f.write(f"Longest contig:          {stats['longest_contig']:,} bp\n")
        f.write(f"Shortest contig:         {stats['shortest_contig']:,} bp\n")
        f.write(f"Mean contig length:      {stats['mean_length']:,.1f} bp\n")
        f.write(f"N50:                     {stats['n50']:,} bp\n")
        f.write(f"Assembly time:           {timing['assembly_time']:.2f} seconds\n\n")
        
        f.write("-"*80 + "\n")
        f.write("TOP 20 LONGEST CONTIGS\n")
        f.write("-"*80 + "\n")
        for i, length in enumerate(contig_lengths[:20], 1):
            f.write(f"{i:3d}. {length:10,} bp\n")
        f.write("\n")
        
        f.write("-"*80 + "\n")
        f.write("CONTIG LENGTH DISTRIBUTION\n")
        f.write("-"*80 + "\n")
        bins = [
            (">100kb", sum(1 for x in contig_lengths if x > 100000)),
            (">50kb", sum(1 for x in contig_lengths if x > 50000)),
            (">10kb", sum(1 for x in contig_lengths if x > 10000)),
            (">5kb", sum(1 for x in contig_lengths if x > 5000)),
            (">1kb", sum(1 for x in contig_lengths if x > 1000)),
            (">500bp", sum(1 for x in contig_lengths if x > 500)),
        ]
        for bin_name, count in bins:
            f.write(f"Contigs {bin_name:8s}:     {count:,}\n")
        f.write("\n")
        
        f.write("-"*80 + "\n")
        f.write("TIMING SUMMARY\n")
        f.write("-"*80 + "\n")
        total_time = timing['total_time']
        f.write(f"Read time:               {timing['read_time']:8.2f} seconds "
                f"({timing['read_time']/total_time*100:5.1f}%)\n")
        f.write(f"Graph construction:      {timing['graph_time']:8.2f} seconds "
                f"({timing['graph_time']/total_time*100:5.1f}%)\n")
        f.write(f"Assembly:                {timing['assembly_time']:8.2f} seconds "
                f"({timing['assembly_time']/total_time*100:5.1f}%)\n")
        f.write(f"Total time:              {total_time:8.2f} seconds "
                f"({total_time/60:.2f} minutes)\n\n")
        
        f.write("="*80 + "\n")
        f.write("END OF REPORT\n")
        f.write("="*80 + "\n")


def assemble_mouse_genome(
    input_file="data/mouse_SE_150bp.fq",
    output_fasta="mouse_assembly.fasta", 
    stats_file="mouse_assembly_stats.txt",
    k_mer_size = 75,
    random_seed = None
):
    """Main driver function for mouse genome assembly.
    
    This function orchestrates the complete assembly pipeline:
    1. Load FASTQ reads
    2. Build De Bruijn graph
    3. Assemble contigs
    4. Calculate statistics
    5. Write output files
    
    Returns:
        dict: Dictionary containing assembly results including:
            - dbg: DeBruijnGraph object
            - contigs: List of assembled sequences
            - stats: Assembly statistics dictionary
            - timing: Performance timing information
    """
    print("="*80)
    print("MOUSE GENOME ASSEMBLY PIPELINE")
    print("="*80)
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print()
    
    timing = {}
    
    print("STEP 1: Loading sequencing reads")
    print("-"*80)
    print(f"Input file: {input_file}")
    print(f"Expected: 10 million reads, 150bp each")
    print()
    
    start_time = time.time()
    reads = read_fastq(input_file)
    timing['read_time'] = time.time() - start_time
    
    num_reads = len(reads)
    total_bases = sum(len(read) for read in reads)
    avg_read_length = total_bases / num_reads if num_reads > 0 else 0
    
    print(f"Reads loaded: {num_reads:,}")
    print(f"Total bases: {total_bases:,} bp")
    print(f"Average read length: {avg_read_length:.1f} bp")
    print(f"Time elapsed: {timing['read_time']:.2f} seconds")
    
    if num_reads > 0:
        print(f"\nSample reads:")
        print(f"  First: {reads[0][:80]}...")
        print(f"  Last:  {reads[-1][:80]}...")
    print()
    
    print("STEP 2: Building De Bruijn graph")
    print("-"*80)
    print(f"K-mer size: {k_mer_size}")
    print(f"Building graph from {num_reads:,} reads...")
    print()
    
    start_time = time.time()
    dbg = DeBruijnGraph(reads, k=k_mer_size)
    timing['graph_time'] = time.time() - start_time
    
    # Calculate graph statistics
    num_nodes = len(dbg.graph)
    num_edges = sum(len(neighbors) for neighbors in dbg.graph.values())
    avg_degree = num_edges / num_nodes if num_nodes > 0 else 0
    
    print(f"Graph construction complete!")
    print(f"  Nodes (unique {k_mer_size-1}-mers): {num_nodes:,}")
    print(f"  Edges (k-mer transitions): {num_edges:,}")
    print(f"  Average out-degree: {avg_degree:.2f}")
    print(f"Time elapsed: {timing['graph_time']:.2f} seconds")
    print()
    
    print("STEP 3: Assembling contigs")
    print("-"*80)
    print(f"Finding connected components and traversing graph...")
    print(f"Random seed: {random_seed} (for reproducibility)")
    print()
    
    start_time = time.time()
    contigs = dbg.assemble_contigs(seed=random_seed)
    timing['assembly_time'] = time.time() - start_time
    
    print(f"Assembly complete!")
    print(f"  Contigs generated: {len(contigs):,}")
    print(f"Time elapsed: {timing['assembly_time']:.2f} seconds")
    print()
    
    print("STEP 4: Calculating assembly statistics")
    print("-"*80)
    
    stats = dbg.get_assembly_stats(contigs)
    
    print(f"Assembly Statistics:")
    print(f"  Number of contigs:     {stats['num_contigs']:,}")
    print(f"  Total assembly length: {stats['total_length']:,} bp")
    print(f"  Longest contig:        {stats['longest_contig']:,} bp")
    print(f"  Shortest contig:       {stats['shortest_contig']:,} bp")
    print(f"  Mean contig length:    {stats['mean_length']:,.1f} bp")
    print(f"  N50:                   {stats['n50']:,} bp")
    print()
    
    # Display distribution of contig lengths
    contig_lengths = sorted([len(c) for c in contigs], reverse=True)
    
    print(f"Contig Length Distribution:")
    print(f"  Top 10 longest contigs:")
    for i, length in enumerate(contig_lengths[:10], 1):
        print(f"    {i:2d}. {length:,} bp")
    print()
    
    # Calculate coverage estimate
    genome_size_estimate = 2700000000  # Mouse genome ~2.7 Gbp
    coverage_estimate = (num_reads * avg_read_length) / genome_size_estimate
    assembly_fraction = (stats['total_length'] / genome_size_estimate) * 100
    
    print(f"Genome Coverage Analysis:")
    print(f"  Mouse genome size (expected): ~{genome_size_estimate:,} bp")
    print(f"  Estimated sequencing coverage: {coverage_estimate:.1f}x")
    print(f"  Assembly size vs. genome: {assembly_fraction:.1f}%")
    print()
    
    print("STEP 5: Writing output files")
    print("-"*80)
    
    # Write assembled contigs to FASTA
    dbg.write_fasta(contigs, output_fasta)
    print(f"✓ Contigs written to: {output_fasta}")
    
    timing['total_time'] = (timing['read_time'] + timing['graph_time'] + 
                           timing['assembly_time'])    # Write detailed statistics
    write_statistics_file(
        stats_file,
        input_file,
        num_reads,
        avg_read_length,
        k_mer_size,
        random_seed,
        num_nodes,
        num_edges,
        stats,
        contig_lengths,
        timing,
        coverage_estimate,
        assembly_fraction
    )
    
    print(f"✓ Statistics written to: {stats_file}")
    print()
    
    print("="*80)
    print("ASSEMBLY COMPLETE")
    print("="*80)
    print(f"Total time: {timing['total_time']:.2f} seconds "
          f"({timing['total_time']/60:.2f} minutes)")
    print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print()
    
    print(f"Summary:")
    print(f"  • Processed {num_reads:,} reads ({total_bases:,} bp)")
    print(f"  • Built graph with {num_nodes:,} nodes and {num_edges:,} edges")
    print(f"  • Assembled {stats['num_contigs']:,} contigs")
    print(f"  • Total assembly: {stats['total_length']:,} bp (N50: {stats['n50']:,} bp)")
    print(f"  • Output files: {output_fasta}, {stats_file}")
    print()
    
    return {
        'dbg': dbg,
        'contigs': contigs,
        'stats': stats,
        'timing': timing,
        'graph_stats': {
            'num_nodes': num_nodes,
            'num_edges': num_edges,
            'avg_degree': avg_degree
        },
        'coverage': coverage_estimate
    }


print("\n" + "="*80)
print("MOUSE GENOME DE BRUIJN GRAPH ASSEMBLY")
print("Student Assignment Driver Program")
print("="*80 + "\n")

# Run the assembly pipeline
result = assemble_mouse_genome()

# Display sample contigs
print("="*80)
print("SAMPLE ASSEMBLED CONTIGS")
print("="*80 + "\n")

contigs = result['contigs']
for i in range(min(5, len(contigs))):
    contig = contigs[i]
    preview = contig[:100] + "..." if len(contig) > 100 else contig
    print(f">contig_{i+1} length={len(contig)}")
    print(preview)
    print()

print("="*80)
print("✓ ASSEMBLY COMPLETE")
print("="*80)
print(f"\nResults saved to:")
print(f"  • mouse_assembly.fasta - {result['stats']['num_contigs']:,} assembled contigs")
print(f"  • mouse_assembly_stats.txt - Detailed assembly statistics")
print(f"\nKey metrics:")
print(f"  • Total assembly: {result['stats']['total_length']:,} bp")
print(f"  • N50: {result['stats']['n50']:,} bp")
print(f"  • Longest contig: {result['stats']['longest_contig']:,} bp")
print(f"  • Coverage: {result['coverage']:.1f}x")
print()


MOUSE GENOME DE BRUIJN GRAPH ASSEMBLY
Student Assignment Driver Program

MOUSE GENOME ASSEMBLY PIPELINE
Start time: 2026-02-22 16:34:05

STEP 1: Loading sequencing reads
--------------------------------------------------------------------------------
Input file: data/mouse_SE_150bp.fq
Expected: 10 million reads, 150bp each

Reads loaded: 100,000
Total bases: 15,000,000 bp
Average read length: 150.0 bp
Time elapsed: 0.17 seconds

Sample reads:
  First: AATCAGGGAGAGACTGGGAGAAGTGGAGGGAGTGGAAATCACAGGAGGGATGTAATATATGAGAGAAGAATAAATGTAAG...
  Last:  TGGTAATTTTCAGAGGTTATCAGTCTTGATATAGAAATGGATACTTTTCTATATTTTATGAAAAATTATTCTATTTTTAA...

STEP 2: Building De Bruijn graph
--------------------------------------------------------------------------------
K-mer size: 75
Building graph from 100,000 reads...

read 99000Graph construction complete!
  Nodes (unique 74-mers): 7,372,507
  Edges (k-mer transitions): 7,374,741
  Average out-degree: 1.00
Time elapsed: 20.40 seconds

STEP 3: Assembling contigs
-